# Deliverable 4 — Entrenamiento QLoRA Multi-Agente (`Enrutador`, `Recuperador`, `Redactor`) + Publicación HF
Notebook oficial de entrenamiento para **Deliverable 4 (`3-LoRA`)**:
- Base única: `Qwen/Qwen3-4B` cuantizado 4-bit NF4 (`bf16` en A100/L4, `fp16` en T4).
- **3 adaptadores independientes entrenados por rol** mediante `scripts/entrenar_agente.py` (`completion_only_loss = True`, `LoRA r=16, alpha=32, dropout=0.05`, `0` ejemplos truncados):
  - `enrutador`: `210` train / `36` val (`max_seq_length=2048`, `eval_steps=10`).
  - `recuperador`: `230` train / `39` val (`max_seq_length=2048`, `eval_steps=15`).
  - `redactor`: `180` train / `30` val (`max_seq_length=4096`, `eval_steps=15`).
- Incluye el **Paso 5 incremental (`publicar_adaptadores_hf.py`)** para publicar `enrutador` y `recuperador` desde Google Drive a Hugging Face sin reentrenar (`redactor` ya publicado en `Pablox0333/qwen3-4b-redactor-udec-d4` @ `739b031`).

In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "-L"], check=False)
%pip -q install -U transformers==5.23.0 peft==0.21.3 trl bitsandbytes==0.50.2 accelerate==1.15.0 datasets matplotlib huggingface-hub==1.33.0

import torch

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    bf16_ok = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | bf16={bf16_ok}")

## Paso 2 — Obtener código de `deliverable-4` y verificar datasets antifuga (725 pares)

In [ ]:
import hashlib
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Pablox0333/Proyecto_GenIA.git"
RAMA = "deliverable-4"
WORKDIR = Path("/content/Proyecto_GenIA")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", RAMA], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "checkout", RAMA], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", RAMA], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, REPO_URL, str(WORKDIR)], check=True)

D4_DIR = WORKDIR / "Deliverable 4"
SCRIPT_ENTRENAR = D4_DIR / "scripts" / "entrenar_agente.py"
SCRIPT_PUBLICAR = D4_DIR / "scripts" / "publicar_adaptadores_hf.py"
assert SCRIPT_ENTRENAR.is_file() and SCRIPT_PUBLICAR.is_file()

print("Commit clonado:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())
for rol in ("enrutador", "recuperador", "redactor"):
    tr = D4_DIR / "datos" / f"{rol}_train.jsonl"
    va = D4_DIR / "datos" / f"{rol}_val.jsonl"
    ntr = sum(1 for l in tr.read_text(encoding="utf-8").splitlines() if l.strip())
    nva = sum(1 for l in va.read_text(encoding="utf-8").splitlines() if l.strip())
    print(f"[{rol:<11}] {ntr} train / {nva} val | sha_tr={hashlib.sha256(tr.read_bytes()).hexdigest()[:12]} sha_va={hashlib.sha256(va.read_bytes()).hexdigest()[:12]}")

## Paso 3 — Entrenar uno o los 3 agentes con `scripts/entrenar_agente.py`
- Edita `AGENTES_A_ENTRENAR` según necesites reentrenar todos (`["enrutador", "recuperador", "redactor"]`) o uno solo.

In [ ]:
from pathlib import Path
import subprocess

AGENTES_A_ENTRENAR = ["enrutador", "recuperador", "redactor"]
USAR_DRIVE = True

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/UdeC_GenIA_D4_3LoRA/entrenamiento")
else:
    OUT_ROOT = Path("/content/d4_3lora_entrenamiento")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

for agente in AGENTES_A_ENTRENAR:
    out_agente = OUT_ROOT / agente
    cmd = [
        "python3",
        str(SCRIPT_ENTRENAR),
        "--agente", agente,
        "--output-dir", str(out_agente),
    ]
    print(f"\n=== Entrenando {agente} -> {out_agente} ===")
    subprocess.run(cmd, check=True)

## Paso 4 — Inspeccionar curvas de pérdida y `smoke_test_val.json` por rol

In [ ]:
import json
from IPython.display import Image, display

for agente in AGENTES_A_ENTRENAR:
    out_agente = OUT_ROOT / agente
    mpath = out_agente / "training_metrics.json"
    if not mpath.is_file():
        continue
    m = json.loads(mpath.read_text(encoding="utf-8"))
    print(f"\n[{agente}] {m['hardware']['gpu']} | resultados={json.dumps(m['resultados'], ensure_ascii=False)}")
    png = out_agente / "loss_curve.png"
    if png.is_file():
        display(Image(filename=str(png)))

## Paso 5 — Publicar `Enrutador` y `Recuperador` en Hugging Face (desde Drive, sin reentrenar)
El `Redactor` ya vive públicamente en `Pablox0333/qwen3-4b-redactor-udec-d4` (`739b031`). Esta celda sube `enrutador` y `recuperador` usando `scripts/publicar_adaptadores_hf.py` leyendo los pesos existentes en Drive.

In [ ]:
import os
import subprocess
from google.colab import drive
from huggingface_hub import notebook_login

drive.mount("/content/drive")
HF_USERNAME = "Pablox0333"
DRIVE_EXP_ENTRENAMIENTO = "/content/drive/MyDrive/UdeC_GenIA_Exp_MultiLoRA/entrenamiento"

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

if token:
    os.environ["HF_TOKEN"] = token
else:
    notebook_login()

cmd_pub = [
    "python3",
    str(SCRIPT_PUBLICAR),
    "--hf-username", HF_USERNAME,
    "--roles", "enrutador,recuperador",
    "--drive-dir", DRIVE_EXP_ENTRENAMIENTO,
]
print("Ejecutando:", " ".join(cmd_pub))
subprocess.run(cmd_pub, check=True)

## Paso 6 — Empaquetar artefactos livianos (`training_metrics.json`, curvas y smoke tests)

In [ ]:
import zipfile
from google.colab import files

bundle_zip = OUT_ROOT / "D4_entrenamiento_3lora_artefactos.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rol in AGENTES_A_ENTRENAR:
        base = OUT_ROOT / rol
        for rel in ("training_metrics.json", "loss_curve.png", "loss_curve.pdf", "smoke_test_val.json", "adapter/README.md"):
            p = base / rel
            if p.is_file():
                zf.write(p, arcname=f"{rol}/{rel}")

print("Paquete generado:", bundle_zip)
files.download(str(bundle_zip))